# NER with CRFs and BERT

In this notebook we tackle **Named Entity Recognition (NER)** on the CoNLL 2002 Spanish corpus using two approaches:

1. **CRF** (Conditional Random Field) with hand-crafted features — `sklearn-crfsuite`
2. **BERT fine-tuning** with a pretrained Spanish model — `transformers`

Both models are evaluated on the same test set, so results are directly comparable.

### Key difference from HMMs

| Model | Type | Features |
|---|---|---|
| HMM | Generative $P(x,y)$ | Word identity only |
| CRF | Discriminative $P(y\mid x)$ | Arbitrary hand-crafted features |
| BERT | Discriminative $P(y\mid x)$ | Learned from data, no engineering |

## Installation

In [ ]:
!pip install sklearn-crfsuite transformers datasets seqeval torch accelerate

## Data Loading

In [ ]:
import random
random.seed(42)
import nltk
nltk.download('conll2002', quiet=True)

from nltk.corpus import conll2002

train_sents = list(conll2002.iob_sents('esp.train'))
test_sents  = list(conll2002.iob_sents('esp.testa'))

# Restrict train to speed up computations with BERT
random.shuffle(train_sents)
train_sents_bert = train_sents[:2000]

# With CRF, training is not restricted
train_sents_crf = train_sents

print(f'Training sentences: {len(train_sents)}')
print(f'Test sentences:     {len(test_sents)}')

In [ ]:
# Each sentence is a list of (token, POS, NER) tuples
def get_tokens(sents):
    return [[tok for tok, pos, ner in s] for s in sents]

def get_pos(sents):
    return [[pos for tok, pos, ner in s] for s in sents]

def get_labels(sents):
    return [[ner for tok, pos, ner in s] for s in sents]

train_tokens_crf = get_tokens(train_sents_crf)
train_pos_crf    = get_pos(train_sents_crf)
train_labels_crf = get_labels(train_sents_crf)

train_tokens_bert = get_tokens(train_sents_bert)
train_pos_bert    = get_pos(train_sents_bert)
train_labels_bert = get_labels(train_sents_bert)

test_tokens  = get_tokens(test_sents)
test_pos     = get_pos(test_sents)
test_labels  = get_labels(test_sents)

In [ ]:
print(train_sents[2])

# Part 1: CRF with hand-crafted features

A CRF models the conditional probability of the label sequence given the input:

$$P(y \mid x) = \frac{1}{Z(x)} \exp\left( \sum_{t} \sum_{k} \lambda_k f_k(y_{t-1}, y_t, x, t) \right)$$

The $f_k$ are **feature functions** that can look at the entire input sequence $x$. This is the main advantage over HMMs: features are not limited to the current word.

## Feature engineering

For each token we extract a dictionary of features. Features capture:
- The token itself and its shape (case, digits, punctuation)
- A context window of surrounding tokens
- Beginning/end of sentence indicators

In [ ]:
def word_features(tokens, pos_tags, i):
    """Feature dictionary for token at position i."""
    tok = tokens[i]
    pos = pos_tags[i]

    features = {
        'bias':           1.0,
        'word.lower':     tok.lower(),
        'word.isupper':   tok.isupper(),
        'word.istitle':   tok.istitle(),
        'word.isdigit':   tok.isdigit(),
        'word.prefix2':   tok[:2].lower(),
        'word.prefix3':   tok[:3].lower(),
        'word.suffix2':   tok[-2:].lower(),
        'word.suffix3':   tok[-3:].lower(),
        'pos':            pos,
        'pos[:2]':        pos[:2],
    }

    # Previous token
    if i > 0:
        tok_prev = tokens[i - 1]
        pos_prev = pos_tags[i - 1]
        features.update({
            '-1:word.lower':   tok_prev.lower(),
            '-1:word.istitle': tok_prev.istitle(),
            '-1:word.isupper': tok_prev.isupper(),
            '-1:pos':          pos_prev,
            '-1:pos[:2]':      pos_prev[:2],
        })
    else:
        features['BOS'] = True  # Beginning of sentence

    # Next token
    if i < len(tokens) - 1:
        tok_next = tokens[i + 1]
        pos_next = pos_tags[i + 1]
        features.update({
            '+1:word.lower':   tok_next.lower(),
            '+1:word.istitle': tok_next.istitle(),
            '+1:word.isupper': tok_next.isupper(),
            '+1:pos':          pos_next,
            '+1:pos[:2]':      pos_next[:2],
        })
    else:
        features['EOS'] = True  # End of sentence

    return features

# VERY SIMPLE VERSION!
def word_features(tokens, pos_tags, i):
    tok = tokens[i]
    pos = pos_tags[i]

    features = {
        'bias': 1.0,
        'word.lower()': tok.lower(),
    }
    
    # Palabra anterior
    if i > 0:
        prev = tokens[i - 1]
        features['prev.lower()'] = prev.lower()
    else:
        features['BOS'] = True  # Beginning of Sentence
    
    # Palabra siguiente
    if i < len(tokens) - 1:
        next = tokens[i + 1]
        features['next.lower()'] = next.lower()
    else:
        features['EOS'] = True  # End of Sentence
    
    return features

def sent_to_features(tokens, pos_tags):
    return [word_features(tokens, pos_tags, i) for i in range(len(tokens))]


# Example: features for the first token of the first sentence
example_features = word_features(train_tokens_crf[0], train_pos_crf[0], 0)
print('Features for token:', train_tokens_crf[0][0])
for k, v in example_features.items():
    print(f'  {k:25s} {v}')

In [ ]:
X_train = [sent_to_features(toks, pos) for toks, pos in zip(train_tokens_crf, train_pos_crf)]
X_test  = [sent_to_features(toks, pos) for toks, pos in zip(test_tokens,  test_pos)]

y_train = train_labels_crf
y_test  = test_labels

print(f'Training examples: {len(X_train)} sentences')
print(f'Test examples:     {len(X_test)} sentences')

## Training the CRF

In [ ]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,             # L1 regularization
    c2=0.1,             # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)
print('Training complete.')

## Evaluation

In [ ]:
from sklearn.metrics import classification_report

y_pred_crf = crf.predict(X_test)

# Flatten for sklearn
y_test_flat     = [lab for seq in y_test     for lab in seq]
y_pred_crf_flat = [lab for seq in y_pred_crf for lab in seq]

labels_ne = sorted(set(y_test_flat) - {'O'})

print('CRF: Results on entity tags (excluding O):')
print(classification_report(y_test_flat, y_pred_crf_flat, labels=labels_ne, zero_division=0))

The `classification_report` function computes the metrics:

 Metric | Description |
|---|---|
| **precision** | Of all samples predicted as class X, how many actually were? `TP / (TP + FP)` |
| **recall** | Of all samples that truly belong to class X, how many did we detect? `TP / (TP + FN)` |
| **f1-score** | Harmonic mean of precision and recall. `2 · (P · R) / (P + R)` |
| **support** | Actual number of samples of that class in the test set |

In addition to this, the following summary rows can be found:

| Row | Description |
|---|---|
| **accuracy** | Overall fraction of correctly classified samples |
| **macro avg** | Simple average of each metric across all classes (unweighted by support) |
| **weighted avg** | Average of each metric weighted by each class's support |
| **micro avg** | Aggregates TP, FP and FN across all classes, then computes global metrics. In multiclass classification it equals accuracy |


## Inspecting the model

One advantage of CRFs over neural models is **interpretability**: we can inspect which features and transitions the model learned.

In [ ]:
import pandas as pd

# Top transition weights
trans_df = pd.DataFrame(
    [(label_from, label_to, weight)
     for (label_from, label_to), weight in crf.transition_features_.items()],
    columns=['from', 'to', 'weight']
).sort_values('weight', ascending=False)

print('Top 10 transitions (positive weights = likely):')
print(trans_df.head(10).to_string(index=False))
print()
print('Bottom 10 transitions (negative weights = unlikely):')
print(trans_df.tail(10).to_string(index=False))

In [ ]:
# Top state features (word -> label associations)
state_df = pd.DataFrame(
    [(attr, label, weight)
     for (attr, label), weight in crf.state_features_.items()],
    columns=['feature', 'label', 'weight']
).sort_values('weight', ascending=False)

print('Top 15 state features:')
print(state_df.head(15).to_string(index=False))

## Exercise

In this code, a very basic implementation of the `word_features` function has been provided. Adding extra features to this function allows to achieve micro-F1 of around 0.78.

# Part 2: BERT fine-tuning

BERT processes the entire sentence at once and produces **contextual embeddings**: the representation of each token depends on all surrounding tokens. We add a linear classification head on top and fine-tune the whole model on our NER task.

We use [`dccuchile/bert-base-spanish-wwm-cased`](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased), a BERT model pretrained on Spanish text.

### Tokenization challenge

BERT uses a **wordpiece tokenizer** that may split a single word into several subword tokens. Since our labels are at the word level, we need to decide what label to assign to continuation subwords. The standard approach is to label only the **first subword** of each word and ignore the rest during loss computation and evaluation.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-cased'
MODEL_NAME = 'distilbert-base-multilingual-cased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# Illustrate the subword tokenization issue
example_words  = train_tokens_bert[0]
example_labels = train_labels_bert[0]

print('Original words and labels:')
for w, l in zip(example_words, example_labels):
    print(f'  {w:20s} {l}')

In [ ]:
# Show how BERT tokenizes the same sentence
encoding = tokenizer(example_words, is_split_into_words=True)
tokens   = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
word_ids = encoding.word_ids()

print('\nBERT subword tokens and their word index:')
for tok, wid in zip(tokens, word_ids):
    word = example_words[wid] if wid is not None else '[special]'
    print(f'  {tok:20s}  word_id={wid}  ({word})')

## Dataset preparation

In [ ]:
# Build label vocabulary
all_labels_flat = [lab for seq in train_labels_bert for lab in seq]
unique_labels   = sorted(set(all_labels_flat))
label2id        = {l: i for i, l in enumerate(unique_labels)}
id2label        = {i: l for l, i in label2id.items()}
IGNORE_INDEX    = -100  # PyTorch convention: ignored in loss computation

print(f'Labels: {unique_labels}')
print(f'label2id: {label2id}')

In [ ]:
def tokenize_and_align_labels(tokens_list, labels_list, tokenizer, label2id, max_length=128):
    """
    Tokenize a list of sentences (already split into words) and align NER labels
    to the subword tokens produced by BERT.

    For each word, only the FIRST subword token receives the real label;
    continuation subwords are assigned IGNORE_INDEX so they don't contribute
    to the loss or evaluation.
    """
    encodings = tokenizer(
        tokens_list,
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

    all_label_ids = []
    for i, word_labels in enumerate(labels_list):
        word_ids  = encodings.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(IGNORE_INDEX)       # [CLS] and [SEP]
            elif word_id != prev_word_id:
                label_ids.append(label2id[word_labels[word_id]])  # first subword
            else:
                label_ids.append(IGNORE_INDEX)       # continuation subword
            prev_word_id = word_id
        all_label_ids.append(label_ids)

    import torch
    encodings['labels'] = torch.tensor(all_label_ids)
    return encodings


train_encodings = tokenize_and_align_labels(train_tokens_bert, train_labels_bert, tokenizer, label2id)
test_encodings  = tokenize_and_align_labels(test_tokens,  test_labels,  tokenizer, label2id)

print('Encodings ready.')
print(f'  input_ids shape:  {train_encodings["input_ids"].shape}')
print(f'  labels shape:     {train_encodings["labels"].shape}')

In [ ]:
print(train_encodings["input_ids"][0])
print(tokenizer.convert_ids_to_tokens(train_encodings["input_ids"][0]))
print(train_encodings["labels"][0])

In [ ]:
import torch
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

train_dataset = NERDataset(train_encodings)
test_dataset  = NERDataset(test_encodings)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size:  {len(test_dataset)}')

## Fine-tuning BERT

We load `BertForTokenClassification`, which adds a linear layer on top of BERT's hidden states to predict a label for each token.

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)#.cuda()

print(f'Model loaded: {MODEL_NAME}')
print(f'Number of labels: {len(unique_labels)}')

In [ ]:
def compute_metrics(eval_pred):
    """Compute token-level precision, recall and F1, ignoring IGNORE_INDEX positions."""
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)

    true_labels = [
        [id2label[l] for l in label_row if l != IGNORE_INDEX]
        for label_row in labels
    ]
    true_preds = [
        [id2label[p] for p, l in zip(pred_row, label_row) if l != IGNORE_INDEX]
        for pred_row, label_row in zip(predictions, labels)
    ]

    # Flatten and compute with sklearn
    from sklearn.metrics import classification_report
    y_true = [l for seq in true_labels for l in seq]
    y_pred = [l for seq in true_preds  for l in seq]

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        'precision': report['weighted avg']['precision'],
        'recall':    report['weighted avg']['recall'],
        'f1':        report['weighted avg']['f1-score'],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir='./bert-ner-spanish',
    num_train_epochs=2, # Execute 2 epochs to speed up calculations
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),   # use mixed precision if GPU available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluation

In [ ]:
from sklearn.metrics import classification_report

predictions_output = trainer.predict(test_dataset)
logits             = predictions_output.predictions
label_ids          = predictions_output.label_ids
pred_ids           = np.argmax(logits, axis=-1)

# Align predictions with gold labels (ignore subword continuations)
y_true_bert = [
    id2label[l]
    for label_row in label_ids
    for l in label_row
    if l != IGNORE_INDEX
]
y_pred_bert = [
    id2label[p]
    for pred_row, label_row in zip(pred_ids, label_ids)
    for p, l in zip(pred_row, label_row)
    if l != IGNORE_INDEX
]

labels_ne = sorted(set(y_true_bert) - {'O'})
print('BERT: Results on entity tags (excluding O):')
print(classification_report(y_true_bert, y_pred_bert, labels=labels_ne, zero_division=0))

# Part 3: Comparison

Let's put the results of both models side by side.

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

def summary_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    rows = []
    for label in sorted(set(y_true) - {'O'}):
        r = report.get(label, {})
        rows.append({
            'model':     model_name,
            'label':     label,
            'precision': round(r.get('precision', 0), 3),
            'recall':    round(r.get('recall', 0), 3),
            'f1':        round(r.get('f1-score', 0), 3),
            'support':   int(r.get('support', 0)),
        })
    return rows

rows  = summary_metrics(y_test_flat,  y_pred_crf_flat, 'CRF')
rows += summary_metrics(y_true_bert, y_pred_bert,      'BERT')

comparison = pd.DataFrame(rows).set_index(['label', 'model']).sort_index()
print(comparison.to_string())

In [ ]:
import matplotlib.pyplot as plt

f1_pivot = comparison['f1'].unstack('model')

ax = f1_pivot.plot(kind='bar', figsize=(10, 5), rot=0)
ax.set_title('F1 score by entity type: CRF vs BERT')
ax.set_xlabel('Entity type')
ax.set_ylabel('F1')
ax.set_ylim(0, 1)
ax.legend(title='Model')
plt.tight_layout()
plt.show()

# Reflection

### CRF
- Requires careful **feature engineering**: the quality of features largely determines performance.
- Very **interpretable**: we can inspect transition and state feature weights.
- Fast to train, works well with limited data.
- No notion of word meaning beyond surface form — it doesn't know that *Madrid* and *Barcelona* are similar.

### BERT
- **No feature engineering**: the model learns its own representations from data.
- Captures **contextual meaning**: the same word gets different representations depending on context.
- Benefits from **pretraining** on large corpora: the model already knows a lot about Spanish before seeing any NER examples.
- More expensive to train and run; less interpretable.